In [1]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
import json
from pathlib import Path

import pandas as pd

from datasmith.docker.context import ContextRegistry, DockerContext, Task
from datasmith.notebooks.utils import update_cr

/mnt/sdd1/atharvas/formulacode/datasmith


19:42:01 WARNING  simple_useragent.core: Falling back to historic user agent.


In [15]:
# # Original
# verified_registry_pth = Path("/mnt/sdd1/atharvas/formulacode/datasmith/scratch/context_registry_final_filtered.json")
# verified_repos_pth = Path(
#     "/mnt/sdd1/atharvas/formulacode/terminal-bench/adapters/formulacode/example_task/formulacode-verified.parquet"
# )
# valid_tasks = None # Load all tasks
# registry_path = Path("scratch/formulacode_verified_context_registry.json")
# dataset_path = Path("dataset/formulacode_verified")

# For new dataset
verified_registry_pth = Path("/mnt/sdd1/atharvas/formulacode/datasmith/scratch/context_registry_final_filtered.json")
verified_repos_pth = Path(
    "/mnt/sdd1/atharvas/formulacode/terminal-bench/adapters/formulacode/example_task/filtered_formulacode-new.parquet"
)

valid_tasks = dict(
    map(lambda kv: (eval(kv[0]), kv[1]),
        json.loads(
            Path(
                "/mnt/sdd1/atharvas/formulacode/terminal-bench/adapters/formulacode/example_task/valid_tasks_new_200.json"
            ).read_text()
        ).items(),
    )
)
valid_tasks = dict(filter(lambda item: item[1] > 1.3, valid_tasks.items()))
print(len(valid_tasks))
registry_path = Path("scratch/formulacode_verified_context_registry.json")
dataset_path = Path("dataset/formulacode_verified_new")

21


In [16]:
verified_repos = pd.read_parquet(verified_repos_pth)

if valid_tasks is not None:
    verified_repos = verified_repos[
        verified_repos[["repo_name", "pr_merge_commit_sha"]].apply(tuple, axis=1).isin(valid_tasks)
    ]

registry = update_cr(ContextRegistry.load_from_file(verified_registry_pth))

In [17]:
def get_task(repo_name, base_commit_sha):
    for task in registry.registry:
        repo_name = f"{task.owner}/{task.repo}"
        sha = task.sha
        if repo_name == repo_name and sha == base_commit_sha:
            return task, registry.registry[task]
    return None, None


new_registry = ContextRegistry() if not registry_path.exists() else ContextRegistry.load_from_file(registry_path)

print(len(new_registry.registry))
verified_repos["is_available"] = False
for idx, row in verified_repos.iterrows():
    repo_name = row["repo_name"]
    base_commit_sha = row["pr_base"]["sha"]
    task, context = get_task(repo_name, base_commit_sha)
    if task in new_registry.registry:
        print(f"Skipping {task} as already in registry")
        verified_repos.at[idx, "is_available"] = True
        continue
    if task is not None:
        verified_repos.at[idx, "is_available"] = True
        new_registry.register(task.with_tag("pkg"), context)


print(len(new_registry.registry))
new_registry.save_to_file(registry_path)

19:45:44 INFO     datasmith.docker.context: Context registry saved to scratch/formulacode_verified_context_registry.json


187
Skipping Task(owner='pandas-dev', repo='pandas', sha='9ff14a3ec4259b04b25aa9ce5b23185574c2c771', commit_date=1754239552.0, env_payload='{"dependencies": ["adbc-driver-manager==1.7.0", "adbc-driver-postgresql==1.7.0", "adbc-driver-sqlite==1.7.0", "aiobotocore==2.23.2", "aiohappyeyeballs==2.6.1", "aiohttp==3.12.15", "aioitertools==0.12.0", "aiosignal==1.4.0", "annotated-types==0.7.0", "attrs==25.3.0", "beautifulsoup4==4.13.4", "blosc2==3.6.1", "botocore==1.39.8", "bottleneck==1.5.0", "cachetools==5.5.2", "certifi==2025.7.14", "charset-normalizer==3.4.2", "click==8.2.1", "contourpy==1.3.3", "cramjam==2.11.0", "cycler==0.12.1", "decorator==5.2.1", "defusedxml==0.7.1", "et-xmlfile==2.0.0", "execnet==2.1.1", "fastparquet==2024.11.0", "fonttools==4.59.0", "frozenlist==1.7.0", "fsspec==2025.7.0", "gcsfs==2025.7.0", "google-api-core==2.25.1", "google-auth==2.40.3", "google-auth-oauthlib==1.2.2", "google-cloud-core==2.4.3", "google-cloud-storage==3.2.0", "google-crc32c==1.7.1", "google-resum

In [21]:
# save each dockerfile context in a folder called formulacode_verified/{repo_name}/{sha}/{all files}
def save_context(context: DockerContext, task: Task, task_dir: Path):
    task_dir.mkdir(parents=True, exist_ok=True)
    task_dir.joinpath("Dockerfile").write_text(context.dockerfile_data)
    task_dir.joinpath("entrypoint.sh").write_text(context.entrypoint_data)
    task_dir.joinpath("docker_build_base.sh").write_text(context.base_building_data)
    task_dir.joinpath("docker_build_run.sh").write_text(context.run_building_data)
    task_dir.joinpath("docker_build_env.sh").write_text(context.env_building_data)
    task_dir.joinpath("docker_build_final.sh").write_text(context.final_building_data)
    task_dir.joinpath("docker_build_pkg.sh").write_text(context.building_data)
    task_dir.joinpath("profile.sh").write_text(context.profile_data)
    task_dir.joinpath("run_tests.sh").write_text(context.run_tests_data)
    task_dir.joinpath("task.txt").write_text(repr(task))

# keep track of the current task_dir's already in the dataset_path.
# if we do not overwrite them, we should manually delete the ones not in verified_repos

current_task_dirs = dict()
if dataset_path.exists():
    for repo_dir in dataset_path.iterdir():
        if not repo_dir.is_dir() or "cache" in repo_dir.name:
            continue
        for sha_dir in repo_dir.iterdir():
            if not sha_dir.is_dir() or "cache" in sha_dir.name:
                continue
            current_task_dirs[(repo_dir.name.replace("_", "/"), sha_dir.name)] = False

for _, row in verified_repos.iterrows():
    task_id = row["task_id"]
    repo_name = row["repo_name"]
    base_commit_sha = row["pr_base"]["sha"]
    task, context = get_task(repo_name, base_commit_sha)
    if not task or not context:
        print(f"Skipping {task_id} as context not found")
        continue
    task_dir = dataset_path / repo_name.replace("/", "_") / base_commit_sha
    save_context(context, task, task_dir)
    current_task_dirs[(repo_name, base_commit_sha)] = True

# Now delete the task_dirs not marked as True in current_task_dirs
for (repo_name, sha), is_present in current_task_dirs.items():
    if not is_present:
        task_dir = dataset_path / repo_name.replace("/", "_") / sha
        print(f"Deleting {task_dir} as not in verified repos")
        for file in task_dir.iterdir():
            file.unlink()
        task_dir.rmdir()
        # If the repo_dir is empty, delete it too
        repo_dir = dataset_path / repo_name.replace("/", "_")
        if not any(repo_dir.iterdir()):
            repo_dir.rmdir()

Deleting dataset/formulacode_verified_new/scikit-image_scikit-image/02477b19f500f3cafc0b7d0b4bf7b0a77145917a as not in verified repos
Deleting dataset/formulacode_verified_new/scikit-image_scikit-image/12be1553f2d439fe997052293b3fe8fcdf69f6f6 as not in verified repos
Deleting dataset/formulacode_verified_new/scikit-image_scikit-image/a86701c19edb4163dcfb9e81a25cae744f3a59dc as not in verified repos
Deleting dataset/formulacode_verified_new/scikit-image_scikit-image/da8b381ce6515a6c4f1f3ed560f42a81f3b3fcac as not in verified repos
Deleting dataset/formulacode_verified_new/scikit-image_scikit-image/ba9f5fee025860fdabb180af321fe51f5215607f as not in verified repos
Deleting dataset/formulacode_verified_new/scikit-image_scikit-image/487a3668f9d83686c5081d609c5f8fdc279fa2aa as not in verified repos
Deleting dataset/formulacode_verified_new/scikit-image_scikit-image/e913dd4cc7b658892f40d676c37afc3220c4b201 as not in verified repos
Deleting dataset/formulacode_verified_new/scikit-image_scikit-

In [53]:
dataset_path.parent.joinpath("config.json").write_text(
    json.dumps({"context_registry_path": str(registry_path), "dockerhub_repo": "all", "dockerhub_namespace": None})
)

137